In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive

 24109_RAMAYANA_REEL.mp4  'Colab Notebooks'		   LSTM_Project
 aclImdb.zip		  'Copy of SDE-Sheet(Core).gdoc'   MFC.zip
 BFS.ipynb		   Employers_data.zip		   Model_Better.zip
 Classroom		   FER2013.zip			   reciept_palakkad.jpg


In [ ]:
!unzip "/content/drive/MyDrive/MFC.zip" -d "/content/mfc_files"

Archive:  /content/drive/MyDrive/MFC.zip
   creating: /content/mfc_files/MFC/
  inflating: /content/mfc_files/MFC/negative_model.keras  
  inflating: /content/mfc_files/MFC/neg_tokenizer.pkl  
  inflating: /content/mfc_files/MFC/positive_model.keras  
  inflating: /content/mfc_files/MFC/pos_tokenizer.pkl  


In [ ]:
!pip install gradio

In [ ]:

!unzip "/content/drive/MyDrive/Model_Better.zip" -d "/content/"

Archive:  /content/drive/MyDrive/Model_Better.zip
   creating: /content/Model_Better/
  inflating: /content/Model_Better/negative_model1.keras  
  inflating: /content/Model_Better/neg_tokenizer1.pkl  
  inflating: /content/Model_Better/positive_model1.keras  
  inflating: /content/Model_Better/pos_tokenizer1.pkl  


In [ ]:
import numpy as np
import pickle
import gradio as gr
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# Load models
# -----------------------------
pos_model = load_model("/content/Model_Better/positive_model1.keras", compile=False)
neg_model = load_model("/content/Model_Better/negative_model1.keras", compile=False)

with open("/content/Model_Better/pos_tokenizer1.pkl", "rb") as f:
    pos_tokenizer = pickle.load(f)

with open("/content/Model_Better/neg_tokenizer1.pkl", "rb") as f:
    neg_tokenizer = pickle.load(f)

MAX_LEN = 20

pos_index_word = {v: k for k, v in pos_tokenizer.word_index.items()}
neg_index_word = {v: k for k, v in neg_tokenizer.word_index.items()}


# -----------------------------
# Suggestion Logic
# -----------------------------
def get_suggestions(sentiment, text):
    if text.strip() == "":
        return ["", "", "", "", ""]

    if sentiment == "Positive":
        model = pos_model
        tokenizer = pos_tokenizer
        index_word = pos_index_word
    else:
        model = neg_model
        tokenizer = neg_tokenizer
        index_word = neg_index_word

    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=MAX_LEN-1, padding='pre')

    preds = model.predict(token_list, verbose=0)[0]
    top_indices = preds.argsort()[-5:][::-1]

    suggestions = []
    for idx in top_indices:
        word = index_word.get(idx)
        suggestions.append(word if word else "")

    return suggestions


# -----------------------------
# Append Word
# -----------------------------
def append_word(current_text, word):
    if word == "":
        return current_text
    return current_text + " " + word


# -----------------------------
# Download Function
# -----------------------------
def save_review(text):
    if text.strip() == "":
        return None

    file_path = "generated_review.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    return file_path


# -----------------------------
# UI
# -----------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("## 🎬 Smart Movie Review Autocomplete")
    gr.Markdown("Select sentiment and start typing. Click suggestions to build your review.")

    sentiment = gr.Radio(["Positive", "Negative"], value="Positive", label="Select Sentiment")

    text_box = gr.Textbox(label="Write Review Here", lines=6)

    # Suggestion buttons
    with gr.Row():
        btn1 = gr.Button("")
        btn2 = gr.Button("")
        btn3 = gr.Button("")
        btn4 = gr.Button("")
        btn5 = gr.Button("")

    text_box.change(
        fn=get_suggestions,
        inputs=[sentiment, text_box],
        outputs=[btn1, btn2, btn3, btn4, btn5]
    )

    btn1.click(append_word, [text_box, btn1], text_box)
    btn2.click(append_word, [text_box, btn2], text_box)
    btn3.click(append_word, [text_box, btn3], text_box)
    btn4.click(append_word, [text_box, btn4], text_box)
    btn5.click(append_word, [text_box, btn5], text_box)

    gr.Markdown("### Download Your Review")

    download_btn = gr.Button("Download Review as .txt")
    file_output = gr.File()

    download_btn.click(
        fn=save_review,
        inputs=text_box,
        outputs=file_output
    )

demo.launch()

/tmp/ipykernel_166/1539716750.py:81: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df719aa4368009c5a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
